# 03 · Опоздания, стоимость заказа и доставки

Проверим три вопроса на доставленных заказах. Для каждого покупателя берём **первый наблюдаемый доставленный заказ**, чтобы один человек не попадал в выборку несколько раз. Фильтры по наличию оценки или штату применяем после этого отбора. Все денежные показатели выражены в R$.

До расчёта закрепляем три двусторонние проверки и семейный уровень ошибки 5 %. Двусторонняя проверка учитывает возможность отличия в любом направлении.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sqlalchemy import create_engine

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Используем те же настройки подключения, что и загрузчик CSV.
sys.path.insert(0, str(ROOT / "sql"))
from connection import database_url

engine = create_engine(database_url(ROOT).replace("postgresql://", "postgresql+psycopg2://", 1))
query = """
    SELECT order_id, customer_unique_id, customer_state, order_purchase_timestamp,
           revenue, freight_value, review_score, is_late
    FROM olist.order_mart
    WHERE order_status = 'delivered'
"""
orders = pd.read_sql(query, engine, parse_dates=["order_purchase_timestamp"])
orders["is_late"] = orders["is_late"].astype("boolean")
assert orders["order_id"].is_unique

In [2]:
buyers = orders.sort_values(["order_purchase_timestamp", "order_id"]).drop_duplicates("customer_unique_id")
assert buyers["customer_unique_id"].is_unique
print(f"Доставленных заказов: {len(orders):,}. Первых заказов покупателей: {len(buyers):,}.")

Доставленных заказов: 96,478. Первых заказов покупателей: 93,358.


## 1. Связано ли опоздание с оценкой

**H₀.** Распределение оценок одинаково при доставке вовремя и с опозданием. **H₁.** Распределения различаются. Ожидаем более низкие оценки у опоздавших заказов. Применяем тест Манна — Уитни: шкала упорядоченная, содержит всего пять значений и много совпадений. Для величины различия покажем разрыв в средней оценке на исходной шкале (опоздание минус доставка вовремя).

In [3]:
rated = buyers.dropna(subset=["is_late", "review_score"])
late_scores = rated.loc[rated["is_late"], "review_score"].to_numpy()
on_time_scores = rated.loc[~rated["is_late"], "review_score"].to_numpy()
display(pd.crosstab(rated["is_late"], rated["review_score"], normalize="index").mul(100).round(1))
test_rating = stats.mannwhitneyu(late_scores, on_time_scores, alternative="two-sided")
effect_rating = late_scores.mean() - on_time_scores.mean()

review_score,1.0,2.0,3.0,4.0,5.0
is_late,,,,,
False,6.6,2.7,8.1,20.5,62.1
True,53.9,8.7,10.9,10.1,16.4


## 2. Различается ли средняя сумма товаров в SP и RJ

**H₀.** В Сан-Паулу и Рио-де-Жанейро средняя сумма за товары в первом заказе одинакова. **H₁.** Средние различаются. Сравниваем адреса покупателей в двух крупных штатах. Используем t-тест Уэлча: он оценивает разницу средних при потенциально разных дисперсиях. Стоимость заказа имеет правый хвост, но наблюдений в обеих группах много. Покажем медианы и верхние квантили, а интервал для разности средних оценим повторной выборкой.

In [4]:
states = buyers.loc[buyers["customer_state"].isin(["SP", "RJ"])].dropna(subset=["revenue"])
display(states.groupby("customer_state")["revenue"].describe(percentiles=[.5, .9, .99]).round(2))
sp_values = states.loc[states["customer_state"] == "SP", "revenue"].to_numpy()
rj_values = states.loc[states["customer_state"] == "RJ", "revenue"].to_numpy()
test_states = stats.ttest_ind(sp_values, rj_values, equal_var=False)
effect_states = sp_values.mean() - rj_values.mean()

,count,mean,std,min,50%,90%,99%,max
customer_state,,,,,,,,
RJ,11913.0,142.84,239.63,0.85,90.00,269.66,1004.38,13440.0
SP,39145.0,125.37,184.27,0.85,79.39,248.99,867.17,6499.0


## 3. Связаны ли стоимость товаров и размер платы за доставку

**H₀.** Ранговая корреляция Спирмена равна нулю. **H₁.** Она отличается от нуля. Рассматриваем суммы **на заказ**. Распределения скошены, а зависимость может быть нелинейной, поэтому сравниваем ранги. Коэффициент Спирмена показывает направление и силу монотонной связи.

In [5]:
amounts = buyers[["revenue", "freight_value"]].dropna()
display(amounts.describe(percentiles=[.5, .9, .99]).T.round(2))
revenue = amounts["revenue"].to_numpy()
freight_value = amounts["freight_value"].to_numpy()
test_freight = stats.spearmanr(revenue, freight_value)
effect_freight = test_freight.statistic

,count,mean,std,min,50%,90%,99%,max
revenue,93358.0,137.51,210.71,0.85,86.90,269.80,998.39,13440.00
freight_value,93358.0,22.78,21.57,0.00,17.18,39.32,104.57,1794.96


## Насколько точны оценки эффекта

Строим 95 % процентильные bootstrap-интервалы (1 200 перевыборок, фиксированное зерно). Для разности средних перевыбираем покупателей внутри каждой группы. Для корреляции перевыбираем **пары** «товары и доставка» одного и того же заказа. Это отдельные интервалы для каждого эффекта.

In [6]:
rng = np.random.default_rng(42)
repeats = 1200

def mean_difference_ci(first, second):
    differences = []
    for _ in range(repeats):
        a = rng.choice(first, size=len(first), replace=True)
        b = rng.choice(second, size=len(second), replace=True)
        differences.append(a.mean() - b.mean())
    return np.quantile(differences, [.025, .975])

ci_rating = mean_difference_ci(late_scores, on_time_scores)
ci_states = mean_difference_ci(sp_values, rj_values)
correlations = []
for _ in range(repeats):
    indices = rng.integers(0, len(revenue), size=len(revenue))
    correlations.append(stats.spearmanr(revenue[indices], freight_value[indices]).statistic)
ci_freight = np.quantile(correlations, [.025, .975])

## Результаты трёх проверок

Поправка Бонферрони умножает каждое p-value на три (максимум — 1). Сравниваем **скорректированное** p-value с 0,05. Если SciPy округляет очень малую вероятность до нуля, в таблице указываем верхнюю границу. P-value оценивает совместимость данных с H₀ при допущениях теста.

In [7]:
rows = [
    ["Оценка", "Манна — Уитни (U)", len(late_scores), len(on_time_scores),
     test_rating, effect_rating, "баллы", ci_rating],
    ["Товары, SP − RJ", "Уэлча (t)", len(sp_values), len(rj_values),
     test_states, effect_states, "R$", ci_states],
    ["Товары и доставка", "Спирмена (ρ)", len(revenue), "—",
     test_freight, effect_freight, "коэффициент ρ", ci_freight],
]

def show_p(value):
    return "< 1e-300" if value < 1e-300 else f"{value:.3g}"

result = []
for question, test_name, n1, n2, test, effect, unit, interval in rows:
    adjusted = min(1, 3 * test.pvalue)
    result.append([question, test_name, n1, n2, round(test.statistic, 3), show_p(test.pvalue),
                   show_p(adjusted), round(effect, 3), unit, f"[{interval[0]:.3f}, {interval[1]:.3f}]",
                   "да" if adjusted < .05 else "нет"])
pd.DataFrame(result, columns=["Гипотеза", "Тест", "n₁", "n₂", "Статистика", "p", "p с поправкой",
                              "Эффект", "Единица", "95 % ДИ эффекта", "Отклоняем H₀"])

,Гипотеза,Тест,n₁,n₂,Статистика,p,p с поправкой,Эффект,Единица,95 % ДИ эффекта,Отклоняем H₀
0,Оценка,Манна — Уитни (U),6209,86526,9.700548e+07,< 1e-300,< 1e-300,-2.023,баллы,"[-2.060, -1.983]",да
1,"Товары, SP − RJ",Уэлча (t),39145,11913,-7.325000e+00,2.5e-13,7.51e-13,-17.469,R$,"[-22.383, -12.906]",да
2,Товары и доставка,Спирмена (ρ),93358,—,4.690000e-01,< 1e-300,< 1e-300,0.469,коэффициент ρ,"[0.464, 0.474]",да


## Что это значит для бизнеса

1. **Оценки.** У опоздавших заказов средняя оценка ниже на 2,02 балла (95 % ДИ для разности от −2,06 до −1,98). Разрыв велик относительно шкалы 1–5. Имеет смысл отдельно изучить обращения по задержанным заказам. Оценку оставляют по желанию, и она может отражать также качество товара или работу продавца.
2. **Штаты.** У первого заказа покупателей из SP средняя сумма товаров на 17,47 R$ ниже, чем в RJ (95 % ДИ для разности от −22,38 до −12,91 R$). Разница может быть связана с составом корзины, категориями и различиями аудитории. Одинаковые условия покупки в двух штатах здесь никто не задавал.
3. **Логистика.** Между суммой товаров и платой за доставку положительная ранговая корреляция 0,469 (95 % ДИ от 0,464 до 0,474). Одинаковая сумма товаров может сопровождаться разной платой за доставку. Обе суммы зависят, например, от числа позиций, веса и маршрута.

Выборка ограничена первыми наблюдаемыми доставленными заказами Olist в исторической выгрузке. Магазины, города и периоды могут объединять похожих покупателей. Интервалы учитывают случайный разброс при перевыборке покупателей, а возможное смещение состава аудитории сохраняется.